In [0]:
# =============================================================
# Notebook : 04_crm_silver.py
# Purpose  : Bronze → Silver for CRM customer profiles
# Source   : bronze/batch/crm_profiles/
# Target   : silver/dim_customer/ (Delta)
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

BRONZE_PATH = "abfss://bronze@walmartdata.dfs.core.windows.net/batch/crm_profiles/*/"
SILVER_PATH = "abfss://silver@walmartdata.dfs.core.windows.net/dim_customer/"

df_raw = spark.read.parquet(BRONZE_PATH)
print(f"Raw rows: {df_raw.count():,}")

In [0]:
# ── Schema + enrichment ───────────────────────────────────────
df_clean = (
    df_raw
    .select(
        F.col("customer_id").cast(StringType()),
        F.col("full_name").cast(StringType()),
        F.col("email").cast(StringType()),
        F.col("phone").cast(StringType()),
        F.col("city").cast(StringType()),
        F.col("loyalty_tier").cast(StringType()),
        F.col("loyalty_points").cast(IntegerType()),
        F.to_date("join_date").alias("join_date"),
        F.to_date("last_purchase_date").alias("last_purchase_date"),
        F.col("total_lifetime_spend").cast(DoubleType()),
        F.col("avg_basket_size").cast(DoubleType()),
        F.col("preferred_category").cast(StringType()),
        F.col("preferred_store_id").cast(StringType()),
        F.col("is_active").cast(BooleanType()),
        F.col("source_system").cast(StringType()),
    )
    # ── Derived columns ───────────────────────────────────────
    .withColumn("days_since_last_purchase",
                F.datediff(F.current_date(), F.col("last_purchase_date")))
    .withColumn("days_since_join",
                F.datediff(F.current_date(), F.col("join_date")))
    .withColumn("customer_segment",
                F.when(F.col("total_lifetime_spend") >= 100000, "VIP")
                 .when(F.col("total_lifetime_spend") >= 50000,  "High Value")
                 .when(F.col("total_lifetime_spend") >= 10000,  "Mid Value")
                 .otherwise("Low Value"))
    .withColumn("churn_risk",
                F.when(F.col("days_since_last_purchase") > 180, "High")
                 .when(F.col("days_since_last_purchase") > 90,  "Medium")
                 .otherwise("Low"))
    .withColumn("tenure_segment",
                F.when(F.col("days_since_join") > 1095, "3+ Years")
                 .when(F.col("days_since_join") > 365,  "1-3 Years")
                 .otherwise("< 1 Year"))
    # ── Dedup on customer_id — keep latest ───────────────────
    .dropDuplicates(["customer_id"])
    # ── Filter bad records ────────────────────────────────────
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("email").isNotNull())
    # ── Audit ─────────────────────────────────────────────────
    .withColumn("silver_processed_at", F.current_timestamp())
    .withColumn("pipeline_version", F.lit("crm_bronze_to_silver_v1"))
)

print(f"Clean rows: {df_clean.count():,}")
print("\nCustomer segments:")
df_clean.groupBy("customer_segment").count().orderBy("count", ascending=False).display()

In [0]:
# ── Write Silver Delta + Unity Catalog ───────────────────────
(
    df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("loyalty_tier")
    .save(SILVER_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_silver.dim_customer
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

print(f"✅ walmart_silver.dim_customer registered")
print(f"   Rows: {spark.table('walmart_silver.dim_customer').count():,}")

# Churn risk summary
print("\nChurn risk summary:")
spark.sql("""
    SELECT
        churn_risk,
        loyalty_tier,
        COUNT(*)                        AS customers,
        ROUND(AVG(total_lifetime_spend),2) AS avg_spend
    FROM walmart_silver.dim_customer
    WHERE is_active = true
    GROUP BY churn_risk, loyalty_tier
    ORDER BY churn_risk, avg_spend DESC
""").display()